# 02 – Leakage Check

Goal: check whether `corrective_orders_14d`, `unplanned_stops_14d` and `inspection_interval_days` only use information known at the moment of each reading.

### 1. Load the clean data
Load the cleaned data from Notebook 01 and sort each machine's readings in time order.

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/machine_observations_clean.csv",
                 parse_dates=["observation_timestamp"])
df = df.sort_values(["machine_id", "observation_timestamp"]).reset_index(drop=True)
print(df.shape)

(10334, 15)


### 2. Failure rate for each risky column
**Why these three are risky:** sensor readings are measured by the machine at the moment of the reading. These three columns come from records written by people in other systems (repair orders, stop logs, inspection schedules). They may be written late, changed later, or written because of a failure. So they might contain information that was not known at the moment of the reading.

For each value of the three risky columns, show how many readings have a failure coming (%).
**Look for:** values with a failure rate far above the overall rate (about 5%).

In [8]:
risky = ["corrective_orders_14d", "unplanned_stops_14d", "inspection_interval_days"]

for col in risky:
    summary = df.groupby(col)["fails_within_7d"].agg(rows="count", failure_rate="mean")
    summary["failure_rate"] = (summary["failure_rate"] * 100).round(1)
    print(col)
    print(summary)
    print()

corrective_orders_14d
                       rows  failure_rate
corrective_orders_14d                    
0                      9454           1.2
1                       880          44.4

unplanned_stops_14d
                     rows  failure_rate
unplanned_stops_14d                    
0                    5514           0.9
1                    3231           6.8
2                    1208          13.9
3                     314          17.8
4                      55          21.8
5                      12           8.3

inspection_interval_days
                          rows  failure_rate
inspection_interval_days                    
14                        1833          21.3
30                        8501           1.3



### 3. Do repair orders and inspection interval always appear together?
Count how often each combination of orders (0, 1) and interval (14, 30) appears.
* **Look for:** whether orders = 1 always comes with interval = 14.

In [9]:
pd.crosstab(df["corrective_orders_14d"], df["inspection_interval_days"])

inspection_interval_days,14,30
corrective_orders_14d,,
0,953,8501
1,880,0


### 4. Machine timeline around a failure
Show one machine's readings before and after a failure.
* **Look for:** when the repair order and the 14-day interval appear, compared to the failure.

In [10]:
pd.set_option("display.max_rows", 200)

machine = "M095"
cols = ["observation_timestamp", "vibration_mm_s", "corrective_orders_14d",
        "unplanned_stops_14d", "inspection_interval_days", "fails_within_7d"]

one = df[df["machine_id"] == machine]
one[cols].iloc[30:46]

,observation_timestamp,vibration_mm_s,corrective_orders_14d,unplanned_stops_14d,inspection_interval_days,fails_within_7d
8146,2024-05-26 22:05:00,2.92,0,0,30,0
8147,2024-06-02 21:32:00,2.83,0,0,30,0
8148,2024-06-09 14:44:00,2.99,0,0,30,0
8149,2024-06-14 01:59:00,1.77,0,0,30,0
8150,2024-06-18 06:53:00,2.02,1,3,14,0
8151,2024-06-24 12:41:00,2.45,1,3,14,1
8152,2024-06-30 02:53:00,0.84,1,3,14,0
8153,2024-07-05 05:03:00,0.40,0,1,14,0
8154,2024-07-10 08:48:00,0.49,0,1,14,0
8155,2024-07-14 16:00:00,1.41,0,1,14,0


### 5. Do failures usually come with a repair order?
Find the first warning row of every failure, and check how often a repair order is already shown.
**Look for:** a high % means most failures have a repair order dated just before them, like M095.

In [11]:
# For each row, look at the previous reading of the same machine
df["prev_fail"] = df.groupby("machine_id")["fails_within_7d"].shift(1).fillna(0)

# First warning row of each failure: fails is 1, and the reading before was 0
first_warning = df[(df["fails_within_7d"] == 1) & (df["prev_fail"] == 0)]

print("Number of failures:", len(first_warning))
print("% that already show a repair order:",
      round(first_warning["corrective_orders_14d"].mean() * 100, 1))

Number of failures: 339
% that already show a repair order: 71.4


### 6. Check: could two 1s in a row be two different breakdowns?
Cell 5 counts only the first 1 of each group, so two 1s in a row are treated as one breakdown. If two 1s in a row are more than 7 days apart, they could be two different breakdowns.
**Look for:** 0 pairs more than 7 days apart means the count of 339 breakdowns is fair.

In [12]:
# Are two 1s in a row ever far apart? (could be two different breakdowns)
df["prev_time"] = df.groupby("machine_id")["observation_timestamp"].shift(1)

two_in_a_row = df[(df["fails_within_7d"] == 1) & (df["prev_fail"] == 1)]
gap_days = (two_in_a_row["observation_timestamp"] - two_in_a_row["prev_time"]).dt.days

print("Pairs of 1s in a row:", len(two_in_a_row))
print("Biggest gap between them (days):", gap_days.max())
print("Pairs more than 7 days apart:", (gap_days > 7).sum())

Pairs of 1s in a row: 166
Biggest gap between them (days): 6
Pairs more than 7 days apart: 0


### 7. Decision

| Column | Decision | Reason |
|---|---|---|
| `corrective_orders_14d` | Not used | When there is a repair order, 44.4% of readings have a failure coming. When there is no repair order, only 1.2% do. Also, 71.4% of failures already show a repair order before them, which is unusually high. The repair order may be written because of the breakdown. |
| `inspection_interval_days` | Not used | It is always 14 days when there is a repair order, so it carries the same information. |
| `unplanned_stops_14d` | Used, but needs checking | The failure rate rises step by step with more stops, which looks like a normal warning sign. But stops are also written by people, so their timing should still be confirmed with the data owner (see README). |

**Plan:** build the main model without the two columns that are not used. Then compare it with a model that includes them, to show how much the result depends on them.